# Photon Mosaic: synthetic data walkthrough

Generates a synthetic imaging movie with known ground-truth fluorescence, then runs it
through the full analysis pipeline: fluorescence extraction, ΔF/F, deconvolution, and
neuropil subtraction.

In [ ]:
import numpy as np
import spikeinterface.widgets as sw

import photon_mosaic as pm
import photon_mosaic.widgets as pw

%matplotlib widget

## Generate synthetic imaging data

In [ ]:
rois, imaging, ground_truth = pm.generate_imaging_with_rois(
    num_frames=10000, bleaching_time=600.0, noise_std="poisson", weighted_rois=True, decay_time=0.7, seed=0
)

In [ ]:
rois

In [ ]:
imaging

In [ ]:
w = pw.plot_imaging_series(imaging, backend="ipywidgets", vmax_percentile=99.75)
w.colorbars["imaging"].set_label("Photon counts")
w.figure.tight_layout()  # re-run: the label wasn't there for the widget's own tight_layout() call

In [ ]:
pw.plot_rois(rois, backend="ipywidgets", width_cm=20)

## Create the ROI analyzer

In [ ]:
analyzer = pm.create_roi_analyzer(rois, imaging)

## Extract fluorescence

In [ ]:
fluorescence_ext = analyzer.compute("fluorescence", n_jobs=4)

In [ ]:
fluorescence = fluorescence_ext.get_data(outputs="recording")

In [ ]:
sw.plot_traces(fluorescence, backend="ipywidgets", time_range=[0, 30])

## Compute ΔF/F

In [ ]:
dff_ext = analyzer.compute("df_over_f", n_jobs=4, method="percentile")
df_over_f = dff_ext.get_data(outputs="recording")
sw.plot_traces(df_over_f, backend="ipywidgets", time_range=[30, 60])

## Deconvolution

In [ ]:
deconv_ext = analyzer.compute("deconvolution", n_jobs=4)

In [ ]:
deconvolved = deconv_ext.get_data(outputs="recording")
sw.plot_traces(deconvolved, backend="ipywidgets", time_range=[30, 60])

### Percentile vs. maximin baseline estimation

An aside comparing the two `df_over_f` baseline-estimation methods -- purely for
display below; deconvolution above already used the percentile method.

In [ ]:
dff_ext_maximin = analyzer.compute("df_over_f", n_jobs=4, method="maximin")

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, interactive_output
from IPython.display import display

t = np.arange(fluorescence_ext.get_data().shape[0]) / imaging.sampling_frequency
fig_methods, ax_methods = plt.subplots(2, 1, figsize=(10, 2.5), sharex=True)


def _plot_dff_methods(roi):
    for a in ax_methods:
        a.clear()
    ax_methods[0].plot(t, fluorescence_ext.get_data()[:, roi], c="gray", lw=0.5, label="F")
    ax_methods[0].plot(t, dff_ext.data["f0"][:, roi], c="C0", alpha=0.7, label=r"$F_0$ (percentile)")
    ax_methods[0].plot(t, dff_ext_maximin.data["f0"][:, roi], c="C1", ls="--", alpha=0.7, label=r"$F_0$ (maximin)")
    ax_methods[0].set_ylabel("Fluorescence")
    ax_methods[0].legend(loc="upper right", fontsize=8)

    dff1 = 100 * dff_ext.get_data()[:, roi]
    dff2 = 100 * dff_ext_maximin.get_data()[:, roi]
    ax_methods[1].axhline(0, ls="--", c="k")
    ax_methods[1].plot(t, dff1, c="C0", lw=0.5, alpha=0.7, label="percentile")
    ax_methods[1].plot(t, dff2, c="C1", lw=0.5, ls="--", alpha=0.7, label="maximin")
    ax_methods[1].set_xlim(t[0], t[-1])
    ax_methods[1].set_ylabel(r"$\Delta F/F$ [%]")
    ax_methods[1].set_xlabel("Time (s)")
    ax_methods[1].legend(loc="upper right", fontsize=8)
    fig_methods.tight_layout()


dff_method_dropdown = Dropdown(options=[int(i) for i in rois.roi_ids], description="ROI:")
dff_method_out = interactive_output(_plot_dff_methods, {"roi": dff_method_dropdown})
display(dff_method_dropdown, dff_method_out)

### Reconstructed movies: raw, extracted, denoised

`traces @ ROIs` paints a per-ROI trace back onto the image via the ROIs' own masks --
each ROI's footprint becomes spatially uniform, at whatever that trace's own value is.
`extracted` (same photon-count units as `raw`) isolates the effect of per-ROI spatial
averaging alone -- still temporally noisy. `FluorescenceNode`'s extraction (see its
docstring) is normalized to reconstruct this painting exactly for non-overlapping ROIs;
these ROIs do overlap (see the crosstalk check below), so it's only an approximation
here. `denoised` adds OASIS's temporal denoising on top; since OASIS runs on ΔF/F (not
raw F), its output is converted back to photon-count units via the fitted baseline
`dff_ext.data["f0"]` before painting, so all three panels stay directly comparable.

In [ ]:
raw_masks = rois.get_roi_image_masks()
masks_flat = raw_masks.reshape(rois.get_num_rois(), -1).astype(np.float32)
video_shape = imaging.get_series(epoch_index=0).shape
fluorescence_movie = (fluorescence_ext.get_data() @ masks_flat).reshape(video_shape)
# deconv_ext.data["denoised"] is on the dF/F scale (OASIS runs on dF/F, not raw F) --
# invert dF/F = (F - F0) / F0 to get back to photon counts before painting onto the movie.
denoised_F = deconv_ext.data["denoised"] * dff_ext.data["f0"] + dff_ext.data["f0"]
denoised_movie = (denoised_F @ masks_flat).reshape(video_shape)
fluorescence_imaging = pm.NumpyImaging(fluorescence_movie, sampling_frequency=imaging.sampling_frequency)
denoised_imaging = pm.NumpyImaging(denoised_movie, sampling_frequency=imaging.sampling_frequency)

w = pw.plot_imaging_series(
    {"Raw": imaging, "Extracted": fluorescence_imaging, "Denoised": denoised_imaging},
    backend="ipywidgets",
    width_cm=10,
    vmax_percentile=99.75,
)
for view in ("Raw", "Extracted", "Denoised"):
    w.colorbars[view].set_label("Photon counts")
w.figure.tight_layout()  # re-run: the labels weren't there for the widget's own tight_layout() call

# Extracted and Denoised are on the same (photon-count) scale, so share one contrast range for
# a fair visual comparison. Raw keeps its own (much wider) range: its shot noise spans most of
# that shared range, so the same clipping would saturate Raw to white and hide its structure.
shared_sample = np.concatenate([fluorescence_movie[:100].ravel(), denoised_movie[:100].ravel()])
shared_vmin, shared_vmax = np.percentile(shared_sample, [2.0, 99.75])
for view in ("Extracted", "Denoised"):
    w.global_vmin[view] = shared_vmin
    w.global_vmax[view] = shared_vmax
    w.images[view].set_clim(shared_vmin, shared_vmax)
    w.colorbars[view].update_normal(w.images[view])


### Denoised vs. ground truth (ΔF/F scale)

Validates recovery accuracy: `denoised` (native ΔF/F scale, no `F0` conversion) compared
directly against the true underlying signal, `ground_truth.clean_traces`. `clean_traces`
is exact by construction, unlike a photon-count reconstruction, which would need the true
per-ROI baseline -- not exposed by the generator. Painting both through the same masks on
the same ΔF/F scale makes the comparison direct.

In [ ]:
denoised_dff_movie = (100 * deconv_ext.data["denoised"] @ masks_flat).reshape(video_shape)
ground_truth_movie = (100 * ground_truth.clean_traces @ masks_flat).reshape(video_shape)
denoised_dff_imaging = pm.NumpyImaging(denoised_dff_movie, sampling_frequency=imaging.sampling_frequency)
ground_truth_imaging = pm.NumpyImaging(ground_truth_movie, sampling_frequency=imaging.sampling_frequency)

w2 = pw.plot_imaging_series(
    {"Denoised": denoised_dff_imaging, "Ground truth": ground_truth_imaging},
    backend="ipywidgets",
    width_cm=10,
    vmax_percentile=99.75,
)
for view in ("Denoised", "Ground truth"):
    w2.colorbars[view].set_label(r"$\Delta F/F$ [%]")
w2.figure.tight_layout()  # re-run: the labels weren't there for the widget's own tight_layout() call

# same units for both views -- share one contrast range for a fair comparison.
shared_sample2 = np.concatenate([denoised_dff_movie[:100].ravel(), ground_truth_movie[:100].ravel()])
shared_vmin2, shared_vmax2 = np.percentile(shared_sample2, [2.0, 99.75])
for view in ("Denoised", "Ground truth"):
    w2.global_vmin[view] = shared_vmin2
    w2.global_vmax[view] = shared_vmax2
    w2.images[view].set_clim(shared_vmin2, shared_vmax2)
    w2.colorbars[view].update_normal(w2.images[view])


**Note:** the cell below checks which of the plotted ROIs share pixels with other ROIs.
Unmixed crosstalk between overlapping ROIs can cause small false transients/events in the
traces shown further down.

In [ ]:
masks = rois.get_roi_image_masks() > 0
for i in rois.roi_ids[:3]:
    overlapping = [int(j) for j in rois.roi_ids if j != i and np.any(masks[i] & masks[j])]
    print(f"ROI {i} overlaps with: {overlapping}")

**Note:** inferred ΔF/F is systematically smaller than ground truth below. The video's
`background` (neuropil, out-of-focus light, dark counts) doesn't scale with each ROI's
F0/expression level and isn't subtracted here, which attenuates recovered ΔF/F (see
`generate_imaging_with_rois`'s `background` docs and the neuropil-subtraction section
further down, which corrects for it).

In [ ]:
from ipywidgets import Dropdown, interactive_output
from IPython.display import display

t = np.arange(dff_ext.get_data().shape[0]) / imaging.sampling_frequency
fig_roi, ax_roi = plt.subplots(3, 1, figsize=(10, 4), sharex=True)


def _plot_roi(roi):
    for a in ax_roi:
        a.clear()
    ax_roi[0].plot(t, fluorescence_ext.get_data()[:, roi], lw=0.5, label="F")
    ax_roi[0].plot(t, dff_ext.data["f0"][:, roi], c="#F0E442", label=r"$F_0$")
    ax_roi[0].set_ylabel("Fluorescence")
    ax_roi[0].legend(loc="upper right", fontsize=8)

    ax_roi[1].plot(t, 100 * dff_ext.get_data()[:, roi], c="gray", lw=0.5, alpha=0.7, label=r"inferred $\Delta F/F$")
    ax_roi[1].plot(t, 100 * deconv_ext.data["denoised"][:, roi], c="C0", lw=0.5, alpha=0.7, label="denoised")
    ax_roi[1].plot(t, 100 * ground_truth.clean_traces[:, roi], c="k", lw=0.5, ls="-.", label="ground truth")
    ax_roi[1].set_ylabel(r"$\Delta F/F$ [%]")
    ax_roi[1].legend(loc="upper right", fontsize=8)

    ax_roi[2].plot(t, deconv_ext.get_data()[:, roi], c="C0", lw=0.5, label="deconvolved")
    ax_roi[2].set_xlim(t[0], t[-1])
    spike_times = t[ground_truth.spikes[:, roi] > 0]
    ax_roi[2].vlines(
        spike_times, 0.9, 1.0, transform=ax_roi[2].get_xaxis_transform(), color="k", label="true spikes"
    )
    ax_roi[2].set_ylabel("Activity [a.u.]")
    ax_roi[2].set_xlabel("Time (s)")
    ax_roi[2].legend(loc="upper right", fontsize=8)
    fig_roi.tight_layout()


roi_dropdown = Dropdown(options=[int(i) for i in rois.roi_ids], description="ROI:")
out = interactive_output(_plot_roi, {"roi": roi_dropdown})
display(roi_dropdown, out)

## Neuropil subtraction

Addresses the attenuation noted above: subtracting each ROI's own Suite2p-style
surround neuropil estimate removes most of the `background` term from its
fluorescence trace before ΔF/F normalization, giving a less attenuated recovered ΔF/F.

More generally, neuropil subtraction also removes genuine neuropil *fluctuations*
(out-of-focus activity correlated with nearby cells) -- not modeled here, since this
generator's `background` is spatially uniform and non-fluctuating beyond its own
bleaching decay; the demo below only exercises the constant-attenuation correction.

In [ ]:
analyzer.compute("neuropil", method="surround")
fluorescence_ext_neuropil = analyzer.compute("fluorescence", n_jobs=4, use_neuropil=True)
dff_ext_neuropil = analyzer.compute("df_over_f", n_jobs=4, method="percentile")
deconv_ext_neuropil = analyzer.compute("deconvolution", n_jobs=4)

In [ ]:
def fit_scale(inferred, ground_truth):
    return (inferred * ground_truth).sum(axis=0) / (inferred * inferred).sum(axis=0)


scale_wo_neuropil = fit_scale(dff_ext.get_data(), ground_truth.clean_traces)
scale_w_neuropil = fit_scale(dff_ext_neuropil.get_data(), ground_truth.clean_traces)
print("median ΔF/F scale factor without neuropil (1.0 = no attenuation):", np.median(scale_wo_neuropil))
print("median ΔF/F scale factor with neuropil:", np.median(scale_w_neuropil))

scale_deconv_wo_neuropil = fit_scale(deconv_ext.get_data(), ground_truth.spikes)
scale_deconv_w_neuropil = fit_scale(deconv_ext_neuropil.get_data(), ground_truth.spikes)
print("median deconvolved scale factor without neuropil:", np.median(scale_deconv_wo_neuropil))
print("median deconvolved scale factor with neuropil:", np.median(scale_deconv_w_neuropil))

from ipywidgets import Dropdown, interactive_output
from IPython.display import display

fig_neuropil, ax_neuropil = plt.subplots(figsize=(10, 2))


def _plot_neuropil_roi(roi):
    ax_neuropil.clear()
    ax_neuropil.plot(t, 100 * dff_ext.get_data()[:, roi], lw=0.5, alpha=0.7, label="without neuropil")
    ax_neuropil.plot(t, 100 * dff_ext_neuropil.get_data()[:, roi], lw=0.5, alpha=0.7, label="with neuropil")
    ax_neuropil.plot(t, 100 * ground_truth.clean_traces[:, roi], ls="-.", c="k", label="ground truth")
    ax_neuropil.set_xlim(30, 60)
    ax_neuropil.set_ylabel(r"$\Delta F/F$ [%]")
    ax_neuropil.set_xlabel("Time (s)")
    ax_neuropil.legend(loc="upper right", fontsize=8)
    fig_neuropil.tight_layout()


# default to ROI 11: lowest baseline (F0), no mask overlap -- the clearest single example of
# the neuropil correction. Other ROIs (e.g. 1, which overlaps ROIs 7/14) are still browsable.
neuropil_roi_dropdown = Dropdown(options=[int(i) for i in rois.roi_ids], value=11, description="ROI:")
neuropil_roi_out = interactive_output(_plot_neuropil_roi, {"roi": neuropil_roi_dropdown})
display(neuropil_roi_dropdown, neuropil_roi_out)